<div style="background-color:#007033; color:white; padding:18px 22px; border-radius:6px; margin-bottom:10px;"><h1 style="color:white; margin:0 0 4px 0;">Week 5 — Serving the Model: FastAPI, Docker, AWS Compute</h1><div style="font-size:15px; opacity:0.9;">FraudGuard Case Study · Stage 5 of 10 · INFT 41000, Omar Altrad, Durham College</div></div>

<div style="background-color:#E0F1E8; border-left:5px solid #007033; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#007033;">This week</b><br>Weeks 1–4 built a model and the data feeding it, but nothing outside this notebook can use either. This week wraps the model in a real HTTP API (FastAPI), packages it as a container (Docker), and deploys that container to <b>real AWS compute</b> — the first time this course provisions something that bills by the hour while it runs.</div>

<div style="background-color:#FDE8E6; border-left:5px solid #B22222; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#B22222;">This week creates real billable compute — read this first</b><br>Every AWS week before this one (Week 3, Week 4) only touched S3 storage, which is essentially free at this scale. <b>This week is different.</b> Section 7 deploys a running container to AWS App Runner, which bills by the second while it's running (a few cents at most for this exercise) — but only while it's running. <b>Section 9 (teardown) is not optional</b> — you will delete the deployed service and the container image before finishing this notebook, every time, no exceptions. This is exactly why Week 3 had you set a budget alert.</div>

| # | Section |
|---|---|
| 0 | Setup |
| 1 | Load prior artifacts |
| 2 | From notebook to service |
| 3 | The prediction function |
| 4 | Build the FastAPI app |
| 5 | Package it: Dockerfile |
| 6 | Run it locally with Docker |
| 7 | Deploy to AWS compute |
| 8 | Beyond one container: Compose & Kubernetes |
| 9 | Teardown (required) |
| 10 | Save your work |

<div style="background-color:#FCF3D9; border-left:5px solid #B58800; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#B58800;">Setup</b><br>New packages this week: <code>fastapi</code>, <code>uvicorn</code>, <code>requests</code> — already added to <code>requirements.txt</code>; run <code>pip install -r requirements.txt</code> again before starting. You'll also need <b>Docker Desktop</b> installed and running (Section 6) — see [docker.com/get-started](https://www.docker.com/get-started/) if you don't have it yet. First time in this project? Run <code>00_Setup.ipynb</code> first.</div>

In [1]:
import sys
from pathlib import Path

def find_project_root():
    '''Walk up from the current working directory until we find the
    Course_Project root (identified by having both requirements.txt
    and a src/ folder). Works no matter which VS Code cwd setting is in
    effect for notebooks (fileDirname or workspaceFolder), because either
    way this notebook lives somewhere inside the project tree.'''
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the Course_Project root (a folder containing both "
        "requirements.txt and src/). Open this notebook from inside the project "
        "folder in VS Code (File > Open Folder > Course_Project), or set PROJECT_ROOT "
        "manually, e.g. PROJECT_ROOT = Path(r'C:\path\to\INFT41000\Course_Project')."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"

for required_dir in (DATA_DIR, MODELS_DIR):
    if not required_dir.is_dir():
        raise FileNotFoundError(
            f"Missing folder: {required_dir}\n"
            "Create it first (see the folder-structure instructions in 00_Setup.ipynb) -- "
            "this notebook does not create folders for you."
        )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/srujanappu/Desktop/INF_41000_Project


<>:18: SyntaxWarning: invalid escape sequence '\p'
<>:18: SyntaxWarning: invalid escape sequence '\p'
/var/folders/n7/t0kt8c913yq5hswldjl00s580000gn/T/ipykernel_3822/2913787404.py:18: SyntaxWarning: invalid escape sequence '\p'
  "manually, e.g. PROJECT_ROOT = Path(r'C:\path\to\INFT41000\Course_Project')."


In [2]:
import json
import subprocess
import time
import joblib
import pandas as pd
import requests

print("Setup OK")

Setup OK


## 1. Load Prior Artifacts

In [3]:
MODEL_PATH = MODELS_DIR / "week1_baseline_model.joblib"
PIPELINE_CONFIG_PATH = DATA_DIR / "pipeline_config.json"

for required_path in (MODEL_PATH, PIPELINE_CONFIG_PATH):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing {required_path.name} in {required_path.parent}. Run Week 1 (model) and "
            "Week 4 (pipeline config) first -- this notebook does not regenerate them for you."
        )

model = joblib.load(MODEL_PATH)
with open(PIPELINE_CONFIG_PATH) as f:
    pipeline_config = json.load(f)

MODEL_FEATURES = ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]
print(f"Loaded model: {type(model).__name__}")
print(f"Model features: {MODEL_FEATURES}")

Loaded model: RandomForestClassifier
Model features: ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']


<div style="background-color:#DBF2FF; border-left:5px solid #005291; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#005291;">Why only 5 features here, not Week 4's 7</b><br>Week 4 engineered 7 features, but Week 1's model was trained on only the 5 raw balance/amount columns — the model in production has to be served with <i>exactly</i> the features it was trained on, nothing more or less. If Week 7's fairness work or a future retraining changes that, the API in Section 4 changes with it. This mismatch risk — API code silently drifting out of sync with what the model actually expects — is a real, common production bug, not a hypothetical one.</div>

## 2. From Notebook to Service

A model sitting in `models/week1_baseline_model.joblib` only helps whoever has this notebook
open. Turning it into a **service** means something else — the bank's payment system, a
mobile app, another microservice — can ask it "is this transaction fraud?" over the network,
in real time, without knowing anything about Python, pandas, or scikit-learn. That requires
three layers, each solving a different problem:

| Layer | Tool | Problem it solves |
|---|---|---|
| Web framework | FastAPI | Turns a Python function into an HTTP endpoint with request/response validation |
| Packaging | Docker | Bundles the app + its exact dependencies so it runs identically anywhere |
| Compute | AWS (App Runner) | Runs the container continuously, with a public address, without you managing a server |

This maps directly onto Week 3's architecture diagram: the **Scoring API** and **Compute
Cluster** components, built for real, today.

## 3. The Prediction Function

Before touching FastAPI, write the plain-Python function that does the actual work — this is
the part worth unit-testing on its own, independent of any web framework.

### Exercise 5.1
Write `predict_transaction(transaction, model)`. `transaction` is a dict with (at least) the
5 keys in `MODEL_FEATURES`. Raise a `ValueError` listing any missing keys. Otherwise, build a
one-row DataFrame with the columns in `MODEL_FEATURES` order, call `model.predict_proba`, and
return `{"isFraud": <bool>, "fraud_probability": <float, rounded to 4 decimals>}` using a
0.5 probability threshold.

In [4]:
def predict_transaction(transaction, model):
    missing = [f for f in MODEL_FEATURES if f not in transaction]
    if missing:
        raise ValueError(f"Missing required feature(s): {missing}")

    row = pd.DataFrame([{f: transaction[f] for f in MODEL_FEATURES}], columns=MODEL_FEATURES)
    fraud_probability = float(model.predict_proba(row)[0, 1])
    return {
        "isFraud": fraud_probability >= 0.5,
        "fraud_probability": round(fraud_probability, 4),
    }


sample_transaction = {
    "amount": 2667.70, "oldbalanceOrg": 18983.07, "newbalanceOrig": 0.0,
    "oldbalanceDest": 5474.90, "newbalanceDest": 8142.60,
}
print(predict_transaction(sample_transaction, model))

try:
    predict_transaction({"amount": 100.0}, model)
except ValueError as e:
    print(f"Correctly raised: {e}")

{'isFraud': True, 'fraud_probability': 0.91}
Correctly raised: Missing required feature(s): ['oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']


## 4. Build the FastAPI App

`FastAPI` turns a function like `predict_transaction` into an HTTP endpoint, and `pydantic`
(FastAPI's companion library) validates incoming JSON automatically — a request missing a
field, or sending a string where a number belongs, gets rejected with a clear error before
your code ever runs.

**Create an `api/` folder now**, directly inside `Course_Project` (next to `data/`, `models/`,
`notebooks/`, `src/`) — same manual-creation rule as every other folder in this course; the
cells below only check that it exists.

### Exercise 5.2
Complete the `/predict` endpoint below (the `Transaction` model, `/health` endpoint, and app
setup are done for you). It should build a one-row DataFrame from `transaction.model_dump()`
in `MODEL_FEATURES` order, call `model.predict_proba`, and return the same shape as Exercise
7.1's `predict_transaction`.

In [5]:
APP_CODE = '''
from pathlib import Path

import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

MODEL_FEATURES = ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]
MODEL_PATH = Path(__file__).parent / "week1_baseline_model.joblib"

app = FastAPI(title="FraudGuard Scoring API")
model = joblib.load(MODEL_PATH)


class Transaction(BaseModel):
    amount: float
    oldbalanceOrg: float
    newbalanceOrig: float
    oldbalanceDest: float
    newbalanceDest: float


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(transaction: Transaction):
    row = pd.DataFrame([transaction.model_dump()], columns=MODEL_FEATURES)
    fraud_probability = float(model.predict_proba(row)[0, 1])
    return {
        "isFraud": fraud_probability >= 0.5,
        "fraud_probability": round(fraud_probability, 4),
    }
'''

API_DIR = PROJECT_ROOT / "api"
if not API_DIR.is_dir():
    raise FileNotFoundError(
        f"Missing folder: {API_DIR}\n"
        "Create it first (see the instructions above) -- this notebook does not create "
        "folders for you."
    )

APP_PATH = API_DIR / "app.py"
APP_PATH.write_text(APP_CODE.strip() + "\n")
print(f"Wrote {APP_PATH}")

import ast
ast.parse(APP_PATH.read_text())
print("app.py is syntactically valid Python")

Wrote /Users/srujanappu/Desktop/INF_41000_Project/api/app.py
app.py is syntactically valid Python


## 5. Package It: Dockerfile

A Dockerfile is a recipe for building a container image: start from a base image, copy in
code and dependencies, and declare how to start the app. Note the separate
`requirements-api.txt` below — the Docker image only needs what the *running API* needs
(FastAPI, uvicorn, pandas, scikit-learn, joblib), not your whole dev environment (Jupyter,
matplotlib, boto3). Shipping less means a smaller, faster, more secure image.

### Exercise 5.3
Complete the Dockerfile: fill in the `EXPOSE` port (the API listens on 8000) and the `CMD`
that starts uvicorn bound to `0.0.0.0:8000` (not `127.0.0.1` — inside a container, the app
must listen on all interfaces or nothing outside the container can reach it).

In [6]:
REQUIREMENTS_API = """fastapi>=0.110
uvicorn>=0.29
pandas>=2.0
scikit-learn>=1.3
joblib>=1.3
"""

DOCKERFILE = """FROM python:3.11-slim

WORKDIR /app

COPY requirements-api.txt .
RUN pip install --no-cache-dir -r requirements-api.txt

COPY app.py .
COPY week1_baseline_model.joblib .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

API_DIR = PROJECT_ROOT / "api"
(API_DIR / "requirements-api.txt").write_text(REQUIREMENTS_API)
(API_DIR / "Dockerfile").write_text(DOCKERFILE)
import shutil
shutil.copy(MODEL_PATH, API_DIR / "week1_baseline_model.joblib")

print(f"Wrote {API_DIR / 'requirements-api.txt'}")
print(f"Wrote {API_DIR / 'Dockerfile'}")
print(f"Copied model to {API_DIR / 'week1_baseline_model.joblib'}")

required_lines = ["FROM python", "COPY requirements-api.txt", "RUN pip install",
                   "EXPOSE 8000", "CMD"]
for line in required_lines:
    assert line in DOCKERFILE, f"Dockerfile is missing a line containing: {line!r}"
print("\nDockerfile has all required instructions.")

Wrote /Users/srujanappu/Desktop/INF_41000_Project/api/requirements-api.txt
Wrote /Users/srujanappu/Desktop/INF_41000_Project/api/Dockerfile
Copied model to /Users/srujanappu/Desktop/INF_41000_Project/api/week1_baseline_model.joblib

Dockerfile has all required instructions.


## 6. Run It Locally With Docker

From a terminal, **in the `api/` folder** this notebook just created:
```bash
cd api
docker build -t fraudguard-api .
docker run -p 8000:8000 fraudguard-api
```
Leave that terminal running, then test it — either with `curl` from another terminal:
```bash
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"amount": 2667.70, "oldbalanceOrg": 18983.07, "newbalanceOrig": 0.0, "oldbalanceDest": 5474.90, "newbalanceDest": 8142.60}'
```
or with the cell below (safe to run any time — it won't crash the notebook if Docker isn't
running, it just tells you so).

In [7]:
try:
    response = requests.post(
        "http://localhost:8000/predict",
        json={"amount": 2667.70, "oldbalanceOrg": 18983.07, "newbalanceOrig": 0.0,
              "oldbalanceDest": 5474.90, "newbalanceDest": 8142.60},
        timeout=3,
    )
    print(response.status_code, response.json())
except requests.exceptions.ConnectionError:
    print("Could not reach http://localhost:8000 -- is `docker run` from Section 6 still "
          "running in a terminal? This is expected if you haven't started the container yet.")

200 {'isFraud': True, 'fraud_probability': 0.91}


## 7. Deploy to AWS Compute

We use **AWS App Runner**: point it at a container image and it handles the server,
networking, and public HTTPS endpoint — no manually-managed EC2 instances, load balancers, or
Kubernetes clusters to configure. (Week 3's compute-preview table mentioned EC2/SageMaker/
Lambda; ECS and EKS also exist for multi-service architectures, but for a single containerized
API, App Runner is the right-sized tool — the same "pick the simplest thing that fits"
principle from Week 2's environment decision.)

**a. Push your image to Amazon ECR** (Elastic Container Registry — AWS's private Docker
registry), from the `api/` folder, replacing `<account-id>` with your account ID from Week 3:
```bash
aws ecr create-repository --repository-name fraudguard-api --region us-east-1
aws ecr get-login-password --region us-east-1 | docker login --username AWS --password-stdin <account-id>.dkr.ecr.us-east-1.amazonaws.com
docker tag fraudguard-api:latest <account-id>.dkr.ecr.us-east-1.amazonaws.com/fraudguard-api:latest
docker push <account-id>.dkr.ecr.us-east-1.amazonaws.com/fraudguard-api:latest
```

**b. Create the App Runner service** (AWS Console → App Runner → Create service):
- Repository type: **Container registry** → **Amazon ECR** → select the `fraudguard-api`
  image you just pushed
- Deployment trigger: Manual
- Port: `8000` (must match the Dockerfile's `EXPOSE`)
- CPU/memory: the smallest option (0.25 vCPU / 0.5 GB) is enough for this exercise
- Service name: `fraudguard-api`

Creation takes a few minutes. When it's **Running**, copy the **Default domain** URL from the
service's page (looks like `https://xxxxxxxxxx.us-east-1.awsapprunner.com`).

**c. Cost check:** App Runner is **not** part of the always-free tier — it bills per
vCPU-second and GB-second while running. At the smallest size, a few minutes of testing costs
a small fraction of a cent. It is *not* free to leave running indefinitely — that's exactly
what Section 9 exists to prevent.

### Exercise 5.4
Paste your service's Default domain URL below and run the cell to test your real, deployed
API. Like Section 6's cell, this is safe to run even before you've deployed anything.

In [8]:
APP_RUNNER_URL = "https://REPLACE-WITH-YOUR-DEFAULT-DOMAIN.us-east-1.awsapprunner.com"

# NOTE: AWS deployment was skipped (credentials not configured), so this URL is still the
# placeholder. Paste your App Runner Default domain here once the service is Running.
if "REPLACE-WITH-YOUR-DEFAULT-DOMAIN" in APP_RUNNER_URL:
    print("APP_RUNNER_URL is still the placeholder -- deploy to App Runner (Section 7) and "
          "paste your Default domain above to test the live API.")
else:
    try:
        health = requests.get(f"{APP_RUNNER_URL}/health", timeout=10)
        print("GET /health ->", health.status_code, health.json())
        response = requests.post(
            f"{APP_RUNNER_URL}/predict",
            json={"amount": 2667.70, "oldbalanceOrg": 18983.07, "newbalanceOrig": 0.0,
                  "oldbalanceDest": 5474.90, "newbalanceDest": 8142.60},
            timeout=10,
        )
        print("POST /predict ->", response.status_code, response.json())
    except requests.exceptions.RequestException as e:
        print(f"Could not reach {APP_RUNNER_URL}: {e}")

APP_RUNNER_URL is still the placeholder -- deploy to App Runner (Section 7) and paste your Default domain above to test the live API.


## 8. Beyond One Container: Docker Compose & Kubernetes

Section 7 deployed exactly **one** container, which is why App Runner was the right-sized tool
for it. Most real production systems aren't one container — an API usually needs a cache, a
database, a queue, or several more copies of itself for scale. Two tools show up constantly in
production job postings and real engineering teams to manage exactly that:

- **Docker Compose** — describes several related containers (e.g. this API plus a cache) that
  run together **on one machine**, started and stopped as a group with a single command
  (`docker compose up`). It's the natural next step once one `docker run` command stops being
  enough.
- **Kubernetes** — orchestrates containers **across a cluster of many machines**: it restarts a
  crashed container automatically, spreads copies across nodes, and rolls out updates without
  downtime. This is the same category as the ECS/EKS alternatives Section 7 mentioned — the tool
  you'd reach for once App Runner's "one container, one region" model isn't enough.

Neither tool will actually run in this notebook's sandbox (no Docker daemon, no cluster here) —
the skill this section teaches is the one Week 3's architecture diagram also taught: **reading
and writing the real configuration files** that describe a system, even before you have
somewhere to run them. That's a skill you can put to use on day one of a new job.

### Exercise 5.5
**a.** Complete the `docker-compose.yml` below: it defines this week's `api` service (built from
the `api/` folder — the same image as Section 5) alongside a `cache` service (a plain Redis
container — a common companion for an API like this one, even though nothing in `app.py` uses it
yet).

**b.** Complete the Kubernetes manifest below: a `Deployment` (which keeps a set number of copies
— *replicas* — of the API running and restarts any that crash) and a `Service` (which gives
those replicas one stable network address — the same job App Runner did for you automatically in
Section 7).

Run the cell after — it writes both files to `api/` and validates their structure (parses as real
YAML, has the fields a real `docker compose` / `kubectl apply` would require), the same
"validate what the sandbox can check" approach used throughout this project.


In [9]:
COMPOSE_YAML = """services:
  api:
    build: ./api
    ports:
      - "8000:8000"
  cache:
    image: redis:7-alpine
    restart: unless-stopped
"""

K8S_MANIFEST = """apiVersion: apps/v1
kind: Deployment
metadata:
  name: fraudguard-api
spec:
  replicas: 2  # so a crashed pod doesn't take the whole API down
  selector:
    matchLabels:
      app: fraudguard-api
  template:
    metadata:
      labels:
        app: fraudguard-api
    spec:
      containers:
        - name: fraudguard-api
          image: <account-id>.dkr.ecr.us-east-1.amazonaws.com/fraudguard-api:latest
          ports:
            - containerPort: 8000  # must match the Dockerfile's EXPOSE from Section 5
---
apiVersion: v1
kind: Service
metadata:
  name: fraudguard-api-service
spec:
  selector:
    app: fraudguard-api
  ports:
    - port: 80
      targetPort: 8000
  type: LoadBalancer
"""

API_DIR = PROJECT_ROOT / "api"
(API_DIR / "docker-compose.yml").write_text(COMPOSE_YAML)
(API_DIR / "k8s-deployment.yaml").write_text(K8S_MANIFEST)

try:
    import yaml
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "pyyaml"], check=True)
    import yaml

compose_doc = yaml.safe_load(COMPOSE_YAML)
assert "services" in compose_doc, "docker-compose.yml must have a top-level 'services' key"
assert "api" in compose_doc["services"] and "cache" in compose_doc["services"], \
    "expected an 'api' service and a 'cache' service"
assert compose_doc["services"]["cache"]["image"].startswith("redis"), \
    "cache service should use a redis image"
print("docker-compose.yml is valid YAML with the expected services.")

k8s_docs = list(yaml.safe_load_all(K8S_MANIFEST))
kinds = [d["kind"] for d in k8s_docs]
assert "Deployment" in kinds and "Service" in kinds, "expected both a Deployment and a Service"
deployment = next(d for d in k8s_docs if d["kind"] == "Deployment")
assert deployment["spec"]["replicas"] == 2, "expected 2 replicas for redundancy"
container = deployment["spec"]["template"]["spec"]["containers"][0]
assert container["ports"][0]["containerPort"] == 8000, \
    "container port must match the Dockerfile's EXPOSE 8000"
print("k8s-deployment.yaml is valid YAML with a Deployment (2 replicas) and a Service.")

print(f"Wrote {API_DIR / 'docker-compose.yml'}")
print(f"Wrote {API_DIR / 'k8s-deployment.yaml'}")


docker-compose.yml is valid YAML with the expected services.
k8s-deployment.yaml is valid YAML with a Deployment (2 replicas) and a Service.
Wrote /Users/srujanappu/Desktop/INF_41000_Project/api/docker-compose.yml
Wrote /Users/srujanappu/Desktop/INF_41000_Project/api/k8s-deployment.yaml


<div style="background-color:#FDE8E6; border-left:5px solid #B22222; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#B22222;">Section 9 — Teardown (required, not optional)</b><br>Complete every step below <b>before you consider this notebook finished</b>, even if you're stopping partway through the exercises above:<br>1. AWS Console → App Runner → select <code>fraudguard-api</code> → <b>Delete</b> (not just "Pause" — Delete removes the service entirely).<br>2. AWS Console → ECR → select the <code>fraudguard-api</code> repository → <b>Delete</b> the image(s), or delete the whole repository.<br>3. Locally: <code>docker ps</code> — if the Section 6 container is still running, <code>docker stop</code> it.<br>4. AWS Console → Billing → Cost Explorer — confirm no App Runner charges are still accruing (a small one-time charge for the minutes it ran is expected and fine — an <i>ongoing</i> charge means teardown isn't finished).</div>

Set this to `True` only after every step above is actually done — this is a self-check, not something this notebook can verify for you.

In [10]:
TEARDOWN_CONFIRMED = False  # change to True only after completing every step in Section 9

if not TEARDOWN_CONFIRMED:
    print("WARNING: Teardown not confirmed. Go back and delete your App Runner service and "
          "ECR image now, then set TEARDOWN_CONFIRMED = True and rerun this cell.")
else:
    print("Teardown confirmed.")

## 10. Save Your Work — Needed for Week 6+


In [11]:
deployment_config = {
    "app_path": str(PROJECT_ROOT / "api" / "app.py"),
    "dockerfile_path": str(PROJECT_ROOT / "api" / "Dockerfile"),
    "model_features": MODEL_FEATURES,
    "deployment_target": "AWS App Runner",
    "compose_file": str(API_DIR / "docker-compose.yml"),
    "k8s_manifest_file": str(API_DIR / "k8s-deployment.yaml"),
    "teardown_confirmed": TEARDOWN_CONFIRMED,
}
with open(DATA_DIR / "deployment_config.json", "w") as f:
    json.dump(deployment_config, f, indent=2)

print(f"Saved: {(DATA_DIR / 'deployment_config.json').resolve()}")
print(json.dumps(deployment_config, indent=2))

Saved: /Users/srujanappu/Desktop/INF_41000_Project/data/deployment_config.json
{
  "app_path": "/Users/srujanappu/Desktop/INF_41000_Project/api/app.py",
  "dockerfile_path": "/Users/srujanappu/Desktop/INF_41000_Project/api/Dockerfile",
  "model_features": [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
  ],
  "deployment_target": "AWS App Runner",
  "compose_file": "/Users/srujanappu/Desktop/INF_41000_Project/api/docker-compose.yml",
  "k8s_manifest_file": "/Users/srujanappu/Desktop/INF_41000_Project/api/k8s-deployment.yaml",
  "teardown_confirmed": false
}


<div style="background-color:#E0F1E8; border-left:5px solid #007033; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#007033;">Submit</b><br>This notebook, run top-to-bottom with no errors · <code>predict_transaction</code> (Exercise 5.1) · the completed FastAPI app (Exercise 5.2) · the completed Dockerfile (Exercise 5.3) · a screenshot of your working <code>/predict</code> response, local or deployed · <code>TEARDOWN_CONFIRMED = True</code> with the steps actually completed.<br><br><b>Next week:</b> Week 6 — securing this API: IAM roles instead of hardcoded trust, AWS Secrets Manager, API authentication, and thinking like an attacker.</div>